# 01 — Exploratory Data Analysis: smoking-cessation EMA dataset

**Goal:** understand the dataset well enough to make the leakage-sensitive design decisions for the first baseline pipeline:

1. *Next craving intensity* → linear regression
2. *Is the next craving high?* → logistic regression

Questions this notebook answers:
- What is one row? What is the participant key and the time ordering?
- How regular is the sampling, and how much data is missing or **imputed**?
- What does the craving target look like (scale, distribution, per-person differences)?
- What should "next" mean, and how many usable `(t → next)` pairs does each definition give?
- Which columns are **unsafe** to use as features (future information, follow-up outcomes)?

> ⚠️ **Do not commit this notebook with outputs.** Outputs embed rows of the dataset, which must stay out of the public repo. Clear outputs before committing (or use `nbstripout`).
>
> ⚠️ Everything here is computed on *all* participants, for description only. Feature selection and thresholds must later be fit on **training participants only**.

## 0. Setup and loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadr

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Locate the project root, so this works whether run from the repo root or from notebooks/.
ROOT = Path.cwd()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# The authors' "analytic samples" were preprocessed by them, but they are the unmodified raw input for QuitOS.
DATA_DIR = ROOT / "data" / "raw"


def read_rds(path: Path) -> pd.DataFrame:
    # pyreadr returns {object_name: DataFrame}; an .rds file holds a single unnamed object.
    return next(iter(pyreadr.read_r(path).values()))


s1 = read_rds(DATA_DIR / "analytic_sample_1.rds")
s2 = read_rds(DATA_DIR / "analytic_sample_2.rds")
for frame in (s1, s2):
    frame["id"] = frame["id"].astype(str)

print("sample 1:", s1.shape, "| sample 2:", s2.shape)

## 1. How do the two files relate?

In [ ]:
ids1, ids2 = set(s1["id"]), set(s2["id"])
print(f"participants: {len(ids1)} (sample 1), {len(ids2)} (sample 2)")
print("sample 2 participants are a subset of sample 1:", ids2 <= ids1)
print("sample 2 EMA rows are a subset of sample 1:  ", set(s2["EMA_id"]) <= set(s1["EMA_id"]))

ema_cols = [c for c in s1.columns if c.endswith("_imp")]
merged = s1.merge(s2[["EMA_id"] + ema_cols], on="EMA_id", suffixes=("", "_s2"))
n_diff = sum(int((merged[c] != merged[f"{c}_s2"]).sum()) for c in ema_cols)
print("EMA values that differ between the files on shared rows:", n_diff)

extra = [c for c in s2.columns if c not in s1.columns]
print(f"columns only in sample 2: {len(extra)}; all start with 'pd_' (Fitbit):", all(c.startswith("pd_") for c in extra))

**Interpretation:** sample 2 = sample 1 restricted to the participants with Fitbit data, plus 90 Fitbit-derived columns.
For the EMA-only v1 baseline, **sample 1 is the dataset** (more participants, same EMA values).

## 2. Column inventory and grouping

In [ ]:
df = s1.copy()

COLUMN_GROUPS = {
    "keys_time": ["nr", "id", "EMA_id", "EMA_number", "study_day", "EMA_date", "EMA_hour", "EMA_date_time"],
    "baseline": [
        "age", "sex", "cpd", "job_type", "post_16_edu", "ethnicity", "time_to_first_cig",
        "motivation_to_stop", "past_quit_attempt", "nrt_no_presc", "nrt_presc", "zyban", "varen",
        "ecig", "group_supp", "ind_supp", "helpline", "book", "website", "app", "none", "other",
    ],
    "study_admin": [
        "mpath_data", "fitbit_data", "start_date", "time_first_prompt", "end_date", "time_last_prompt",
        "start_datetime", "end_datetime", "altered_prompt", "altered_prompt_start_time",
        "altered_prompt_study_day", "participant_specific_variable_1", "participant_specific_variable_2",
    ],
    "follow_up_outcomes": ["smoking_fup", "last_time_smoke_fup", "CO_reading", "CO_reading1", "CO_reading2"],
    "ema_time_varying": ema_cols,
    "derived_lapse": ["lapse_prior", "lapse_lagged"],
}

assigned = pd.Series([c for cols in COLUMN_GROUPS.values() for c in cols])
print("unassigned columns:", [c for c in df.columns if c not in set(assigned)])
print("columns assigned twice:", assigned[assigned.duplicated()].tolist())


def column_summary(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    within = frame.groupby("id")[cols].nunique(dropna=False).max()
    return pd.DataFrame({
        "dtype": frame[cols].dtypes.astype(str),
        "pct_null": (frame[cols].isna().mean() * 100).round(1),
        "nunique": frame[cols].nunique(),
        "constant_within_participant": within.eq(1),
    })


for group, cols in COLUMN_GROUPS.items():
    print(f"\n### {group} ({len(cols)} columns)")
    display(column_summary(df, cols))

Notes:
- The `_imp` suffix marks EMA items whose **missing responses were imputed by the dataset authors**, so the values look complete even where no response was given. See section 4.
- `baseline` and `follow_up_outcomes` are constant within each participant.
- Several single-choice items are one-hot encoded (`location_*`, `activity_*`, `social_context_*`). Some of these are constant across the whole dataset (nunique = 1) and carry no information.

## 3. Study design: what is one row?

In [ ]:
# File order follows the per-participant EMA sequence number (EMA_id = "<id>_<seq>").
df["_seq"] = df["EMA_id"].str.split("_").str[1].astype(int)
print("file order is monotonic in _seq within each participant:",
      df.groupby("id")["_seq"].apply(lambda s: s.is_monotonic_increasing).all())
df = df.sort_values(["id", "_seq"]).reset_index(drop=True)

df["scheduled"] = df["EMA_number"].notna()  # NaN EMA_number = self-initiated report
df["observed"] = df["EMA_date_time"].notna()  # NaT = no response timestamp -> imputed row

# Do observed response timestamps increase with the sequence number?
o_tmp = df[df["observed"]].copy()
o_tmp["back_h"] = o_tmp.groupby("id")["EMA_date_time"].diff().dt.total_seconds() / 3600
inversions = o_tmp[o_tmp["back_h"] < 0]
print(f"observed rows whose timestamp is EARLIER than the previous observed row in _seq order: {len(inversions)}")
print("  participants:", sorted(inversions["id"].unique()),
      "| all are self-initiated reports:", bool((~inversions["scheduled"]).all()),
      "| max inversion (min):", round(-inversions["back_h"].min() * 60, 1))

print("\nrows per participant:", df.groupby("id").size().describe()[["min", "50%", "max"]].to_dict())
print("study days:", sorted(df["study_day"].unique()))
print("scheduled prompts per participant-day:", df[df["scheduled"]].groupby(["id", "study_day"]).size().unique())
print("self-initiated rows:", int((~df["scheduled"]).sum()),
      "| all are lapse reports:", bool(df.loc[~df["scheduled"], "lapse_event_imp"].eq(1).all()))
print("participants with an altered prompt schedule:", df.loc[df["altered_prompt"].eq("yes"), "id"].nunique())

fig, ax = plt.subplots(figsize=(10, 3))
df.loc[df["scheduled"], "EMA_hour"].value_counts().sort_index().plot.bar(ax=ax)
ax.set(title="Scheduled prompts by EMA_hour (all participants)", xlabel="hour of day", ylabel="prompts")
plt.show()

**Design:** about 10 study days × 16 hourly prompts per day = 160 scheduled prompts per participant, plus occasional self-initiated lapse reports.
Prompt windows differ between participants (some start at 6–8 am; a few have prompts after midnight).

**Ordering quirk:** the authors' sequence (`EMA_id`) sorts rows by `(EMA_date, EMA_hour)`. In the same hour it puts the scheduled prompt *before* a self-initiated lapse report, even when the report was submitted a few minutes earlier (4 such cases).
⇒ In the pipeline, **order observed rows by `EMA_date_time`**, not by `EMA_id`. Note that the authors' `lapse_prior` / `lapse_lagged` follow `EMA_id` order.

## 4. Missingness and imputation

In [ ]:
sched = df[df["scheduled"]]
resp = sched.groupby("id")["observed"].mean().sort_values()
print("scheduled-prompt response rate per participant:")
print(resp.describe().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
resp.plot.bar(ax=axes[0])
axes[0].set(title="Response rate per participant", ylabel="share observed", ylim=(0, 1))
axes[0].tick_params(axis="x", labelsize=6)
sched.groupby("study_day")["observed"].mean().plot(marker="o", ax=axes[1])
axes[1].set(title="Response rate by study day", ylim=(0, 1))
sched.groupby("EMA_hour")["observed"].mean().plot(marker="o", ax=axes[2])
axes[2].set(title="Response rate by hour", ylim=(0, 1))
plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby("observed")["craving_imp"].agg(["count", "mean", "std"]).round(2))
print("non-integer imputed craving values:", int((df.loc[~df["observed"], "craving_imp"] % 1 != 0).sum()))

bins = np.arange(-0.5, 11.5, 1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist([df.loc[df["observed"], "craving_imp"], df.loc[~df["observed"], "craving_imp"]],
        bins=bins, density=True, label=["observed", "imputed"])
ax.set(title="Craving: observed vs imputed rows", xlabel="craving (0-10)", ylabel="density", xticks=range(11))
ax.legend()
plt.show()

**Why this matters for leakage:**
- Imputed values are rounded integers, so they look just like real answers. **The only way to tell them apart is `EMA_date_time` being NaT.**
- Imputed rows have a narrower craving distribution, which is typical of model-based imputation.
- We don't yet know how the authors imputed. If the method used a participant's *whole* time series (common with multiple imputation), imputed values carry information from the future.
- ⇒ **Targets must come from observed responses only.** Whether imputed rows may be used as *features* is a separate decision (safest for v1: no; use observed rows only, plus "time since last response" features).

## 5. The craving target

In [ ]:
obs = df[df["observed"]].copy()
print(obs["craving_imp"].describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 3.5))
obs["craving_imp"].value_counts().sort_index().plot.bar(ax=ax)
ax.set(title="Observed craving distribution", xlabel="craving (0-10)", ylabel="responses")
plt.show()

In [ ]:
# Candidate "high craving" thresholds. In the pipeline, the chosen threshold must be FIXED a priori
# (or derived from training participants only), never from the full dataset.
per_person = obs.groupby("id")["craving_imp"]
rows = []
for t in range(5, 10):
    share_by_person = per_person.apply(lambda s: (s >= t).mean())
    rows.append({
        "threshold (>=)": t,
        "share_high": round((obs["craving_imp"] >= t).mean(), 3),
        "participants_never_high": int((share_by_person == 0).sum()),
        "participants_always_high": int((share_by_person == 1).sum()),
        "median_person_share_high": round(share_by_person.median(), 3),
    })
pd.DataFrame(rows)

In [ ]:
order = per_person.mean().sort_values().index
fig, ax = plt.subplots(figsize=(14, 4))
ax.boxplot([obs.loc[obs["id"] == pid, "craving_imp"] for pid in order])
ax.set_xticks(range(1, len(order) + 1), order, rotation=90, fontsize=7)
ax.set(title="Observed craving per participant (sorted by mean)", ylabel="craving (0-10)")
plt.show()

between = per_person.mean().var()
within = per_person.var().mean()
print(f"approximate ICC (between-person share of variance): {between / (between + within):.2f}")

**Interpretation:** roughly half of the craving variance is *between* people (ICC ≈ 0.5).
A participant-level split is therefore essential: with a row-level split, a model could score well just by memorising each person's average craving.
It also means that predicting for a *new* participant (cold start) is much harder than predicting for a known one.

## 6. Temporal structure: what should "next" mean?

In [ ]:
obs = obs.sort_values(["id", "EMA_date_time"])
obs["gap_h"] = obs.groupby("id")["EMA_date_time"].diff().dt.total_seconds() / 3600
obs["same_day_as_prev"] = obs.groupby("id")["study_day"].diff().eq(0)

print("hours between consecutive observed responses:")
print(obs["gap_h"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).round(2).to_string())
print("share of consecutive observed pairs on the same study day:", round(obs["same_day_as_prev"].mean(), 3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
obs["gap_h"].clip(upper=14).plot.hist(bins=56, ax=axes[0])
axes[0].set(title="Gap to previous observed response (clipped at 14h)", xlabel="hours")
obs.groupby(obs["EMA_date_time"].dt.hour)["craving_imp"].mean().plot(marker="o", ax=axes[1])
axes[1].set(title="Mean craving by hour of day", xlabel="hour", ylim=(0, 10))
obs.groupby("study_day")["craving_imp"].mean().plot(marker="o", ax=axes[2])
axes[2].set(title="Mean craving by study day", xlabel="study day", ylim=(0, 10))
plt.tight_layout()
plt.show()

In [ ]:
def build_pairs(frame: pd.DataFrame, rule: str) -> pd.DataFrame:
    # Pair each row t with the next row in time within the same participant.
    # Only the *target* comes from the next row; features would come from row t (and earlier).
    if rule == "next_row_any":  # includes imputed rows on both sides
        f = frame.sort_values(["id", "_seq"]).copy()
    else:  # skip unanswered prompts: next OBSERVED response
        f = frame[frame["observed"]].sort_values(["id", "EMA_date_time"]).copy()
    g = f.groupby("id")
    f["y_next"] = g["craving_imp"].shift(-1)
    f["next_time"] = g["EMA_date_time"].shift(-1)
    f["next_day"] = g["study_day"].shift(-1)
    f = f[f["y_next"].notna()]
    f["gap_next_h"] = (f["next_time"] - f["EMA_date_time"]).dt.total_seconds() / 3600
    if rule == "next_observed_same_day":
        f = f[f["next_day"] == f["study_day"]]
    if rule == "next_observed_within_3h":
        f = f[f["gap_next_h"] <= 3]
    return f


rows = []
for rule in ["next_row_any", "next_observed", "next_observed_same_day", "next_observed_within_3h"]:
    p = build_pairs(df, rule)
    rows.append({
        "rule": rule,
        "pairs": len(p),
        "participants": p["id"].nunique(),
        "corr(craving_t, craving_next)": round(np.corrcoef(p["craving_imp"], p["y_next"])[0, 1], 3),
        "MAE persistence (next = current)": round((p["y_next"] - p["craving_imp"]).abs().mean(), 2),
        "MAE global mean": round((p["y_next"] - p["y_next"].mean()).abs().mean(), 2),
    })
pd.DataFrame(rows)

**Interpretation:**
- Craving is strongly autocorrelated (r ≈ 0.7 between consecutive responses). The simple **persistence baseline (next craving = current craving)** is the bar our models need to beat. Reporting only R² or AUC without it would be misleading.
- About 8% of consecutive responses span the overnight gap. Whether "next" may cross midnight is a modelling choice; restricting to same-day pairs gives a cleaner one-hour-ahead task.
- The `next_row_any` rule pairs imputed rows with each other and inflates the correlation. Don't use it.

## 7. Leakage audit of specific columns

In [ ]:
g = df.groupby("id")["lapse_event_imp"]
print("lapse_prior  == PREVIOUS row's lapse_event:", (df["lapse_prior"] == g.shift(1).fillna(0)).mean())
print("lapse_lagged == NEXT row's lapse_event:    ", (df["lapse_lagged"] == g.shift(-1).fillna(0)).mean())
print("lapse_lagged == SAME row's lapse_event:    ", round((df["lapse_lagged"] == df["lapse_event_imp"]).mean(), 3))

| Column(s) | What it is | Use as feature at time *t*? |
|---|---|---|
| `lapse_lagged` | Despite the name, it is the **next** row's lapse (a lead) | ❌ **Future information. Exclude.** |
| `lapse_prior` | Previous row's lapse | ✅ OK (past) |
| `smoking_fup`, `last_time_smoke_fup`, `CO_reading*` | Follow-up / verification outcomes measured after the EMA period | ❌ Exclude (post-outcome) |
| `EMA_date_time` | Response timestamp; NaT means imputed | Only for ordering, gaps and observed masks; not a raw feature |
| `nr`, `id`, `EMA_id`, `EMA_number` | Identifiers / counters | ❌ Not features (`id` is the grouping key for the split) |
| `start_*`, `end_*`, `altered_prompt*` | Study administration | ❌ Not features in v1 |
| `participant_specific_variable_*` | Free-text personalised items, mostly missing | ❌ Not in v1 |
| `*_imp` EMA items at time *t* | Same-moment self-reports | ✅ As features for predicting **t+1**, from observed rows only |
| baseline items (`age`, `cpd`, ...) | Collected at enrolment | ✅ OK (known before the EMA period) |
| `pd_*` (sample 2) | Fitbit window summaries | ⚠️ Later: first confirm each window ends before the prompt |

## 8. Descriptive associations with next craving

Descriptive only, computed on all participants. **Do not use these numbers to select features**, because that would leak test-set information. The point is to sanity-check that same-moment EMA items carry some signal.
Because of the large between-person component, both raw and within-person-centred correlations are shown.

In [ ]:
pairs = build_pairs(df, "next_observed_same_day")
num_ema = [c for c in ema_cols if pairs[c].nunique() > 1]

raw = pairs[num_ema].corrwith(pairs["y_next"])
centred = pairs[num_ema] - pairs.groupby("id")[num_ema].transform("mean")
y_c = pairs["y_next"] - pairs.groupby("id")["y_next"].transform("mean")
within_r = centred.corrwith(y_c)

assoc = pd.DataFrame({"r_raw": raw, "r_within_person": within_r}).round(3)
assoc.reindex(assoc["r_raw"].abs().sort_values(ascending=False).index).head(20)

## 9. Fitbit columns (sample 2): naming only

In [ ]:
pd_cols = [c for c in s2.columns if c.startswith("pd_")]
parts = pd.Series(pd_cols).str.extract(r"pd_(\d)_(hr|step)_(\d)_(.+)")
parts.columns = ["pd", "signal", "window", "stat"]
print(len(pd_cols), "columns")
print({c: sorted(parts[c].unique()) for c in parts.columns})

Out of scope for v1. Before using these, we need the authors' definition of `pd_k` and `window` to confirm each window ends *before* the prompt.

## 10. Summary: decisions for the pipeline

| Topic | Finding | Proposed v1 decision |
|---|---|---|
| Dataset | Sample 2 ⊂ sample 1 (+ Fitbit) | Use **sample 1** (38 participants) |
| Row | One EMA prompt; 16/day × 10 days + self-initiated lapse reports | Order by `id`, then `EMA_date_time` (not `EMA_id`: 4 inversions) |
| Imputation | ~23% of rows imputed by the authors; only detectable via `EMA_date_time` NaT | **Targets: observed only.** Features: observed only in v1 |
| "Next" | Median gap 1h; ~8% of pairs cross the night | Next observed response, **same study day** (gap as a feature) |
| Target scale | Craving 0–10, mean ≈ 4.9, SD ≈ 3.2 | Regression on raw 0–10 |
| "High" | ≥7 → ~38% positive; ≥8 → ~24% | Fix the threshold up front (e.g. ≥7), record it in `config.py` |
| Between-person variance | ICC ≈ 0.5 | Participant-level split; **GroupKFold** CV (only 38 people) |
| Autocorrelation | r ≈ 0.7 | Report **persistence** and **mean** baselines next to the models |
| Leaky columns | `lapse_lagged` = future; follow-up outcomes | Hard-coded exclusion list, plus a test that enforces it |